In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st

In [ ]:
homer_path = ''
env = ''

In [ ]:
os.system(env+'python ./metilene3/metilene3.py \
    -i ./data/GSE186458_celltypes.input.tsv \
    -o ./All \
    -t 16 \
    -n 3 \
    -plot True\
')

In [ ]:
os.system('cp ./All/DMRs-unsupervised.tsv ../SourceData/Fig.ED4.txt')

In [ ]:
color_palette = {}

import random
random.seed(0)
for i in range(100):
    color_palette[i] = (random.randint(0,100)/100,random.randint(0,100)/100,random.randint(0,100)/100)

In [ ]:
colors = pd.read_table('./All/clusters.tsv', index_col=0)
colors['group'] = [color_palette[int(i.split('G')[1])] for i in colors['Group']]
colors.index = [i.replace('Z00000','') for i in colors.index]
colors.head()

In [ ]:
from Bio import Phylo
import matplotlib.pyplot as plt

tree = Phylo.read("./All/DMTree.nwk", "newick")

def change_labels(clade):
    if clade.name:
        clade.name = clade.name.split('=')[-1].replace('Z00000','')
    for subclade in clade.clades:
        change_labels(subclade)

change_labels(tree.root)

cmap = colors['group'].to_dict()
for i in colors.index:
    cmap[i.split('=')[-1].replace('Z00000','')] = cmap[i]
f,a = plt.subplots(figsize=[15,35])
Phylo.draw(tree, axes=a, do_show=False, label_colors=cmap)
plt.xlim([-30,3100])
a.spines['top'].set_visible(False)
a.spines['left'].set_visible(False)
a.spines['right'].set_visible(False)
a.set_xlabel(None)
a.yaxis.set_visible(False)
plt.savefig('./figures/ED4.pdf', bbox_inches='tight')

In [ ]:
dmrs = pd.read_table('./All/DMRs.tsv')
dmrs

In [ ]:
met = pd.read_table('./data/GSE186458_celltypes.input.tsv', na_values='.', \
                    usecols=['chrom','end']+list(pd.read_table('./data/GSE186458_celltypes.input.tsv', na_values='.', nrows=0).columns\
    [pd.read_table('./data/GSE186458_celltypes.input.tsv', na_values='.', nrows=0\
                  ).columns.str.contains('Pancrea')]))
met.columns = [i.split('=')[-1] for i in met.columns]
met.columns = ['chr','pos']+list(met.columns[2:])
met

In [ ]:
from pybedtools import BedTool

def find_overlapping_regions_df(bed1_df, bed2_df, bed1_cols, bed2_cols):
    bed_1 = BedTool.from_dataframe(bed1_df[bed1_cols].sort_values(bed1_cols))
    bed_2 = BedTool.from_dataframe(bed2_df[bed2_cols].sort_values(bed2_cols))
    return BedTool.to_dataframe(bed_1.intersect(bed_2, wa=True, wb=True))

typec = {
        'Alpha':sns.color_palette("Paired")[5],
        'Beta':sns.color_palette("Paired")[9],
        'Delta':sns.color_palette("Paired")[3],
        'Duct':sns.color_palette("Paired")[1],
    }

def plotIGV(tmp, cutoff=0):
    tmp['end'] = tmp['pos']+1
    dmrcpgs = find_overlapping_regions_df(tmp[['chr','pos','end']], dmrs.loc[dmrs['meandiffabs']>=cutoff][['chr','start','stop','meandiffabs','sig.comparison']], \
                                                    ['chr','pos','end'], ['chr','start','stop','meandiffabs','sig.comparison'])
          
    dmrcpgs.index = dmrcpgs['start']

    dmrs_tmp = dmrcpgs['name	score	strand'.split('\t')].drop_duplicates()
    
    for i in range(4):
        tmp[str(i)] = tmp['pos'].map(dmrcpgs['thickEnd'].apply(lambda x:int(x.split('|')[i])).to_dict()).fillna(0)
            
    f,d = plt.subplots(5,1,figsize=[10,5], sharex=True)
    
    for j,k in enumerate(['Alpha','Beta','Delta']):

        lenR = tmp['pos'].max()-tmp['pos'].min()
        sns.lineplot(x=[tmp['pos'].min()-0.01*lenR,tmp['pos'].max()+0.01*lenR],\
                                y=[1,1],color='#f0f0f0', linewidth=1,ax=d[j], zorder=0)
        sns.lineplot(x=[tmp['pos'].min()-0.01*lenR,tmp['pos'].max()+0.01*lenR],\
                                y=[0,0],color='#f0f0f0', linewidth=1,ax=d[j], zorder=0)
        
        sns.scatterplot(x=tmp['pos'],\
                        y=tmp[tmp.columns[tmp.columns.str.contains(k)]].T.mean(),color=typec[k],s=36, linewidth=0,ax=d[j])
        
        d[j].spines['top'].set_visible(False)
        d[j].spines['right'].set_visible(False)
        d[j].spines['bottom'].set_visible(False)
        d[j].xaxis.set_visible(False)
        d[j].set_ylim([-0.1,1.1])
        d[j].set_yticks([0,1])
        d[j].set_ylabel('')

        for dmri,dmr in dmrs_tmp.iterrows():
            d[j].axvspan(dmr['score'], dmr['strand'], alpha=0.1, color='red')
    
    tmp.index = tmp['pos']
    
    tocolor = {'1':'#9dc2a9','2':'#e7e6e6','3':'#e4c198'}
    for j,i in dmrcpgs.drop_duplicates(['score','strand']).iterrows():
        for l,k in enumerate(i['thickEnd'].split('|')):
            sns.lineplot(x=[i['score'],i['strand']],\
                                y=[int(-l)/10+0.8,int(-l)/10+0.8],color=tocolor[k], linewidth=3.9,ax=d[-1])
    plt.ylim([0,1])
    d[-1].spines['top'].set_visible(False)
    d[-1].spines['right'].set_visible(False)
    d[-1].spines['left'].set_visible(False)
    d[-1].spines['bottom'].set_visible(False)
    d[-1].xaxis.set_visible(False)
    d[-1].yaxis.set_visible(False)
    
    print(ab[0]+':'+str(tmp['pos'].min())+'-'+str(tmp['pos'].max()))
    d[0].set_title(ab[0]+':'+str(tmp['pos'].min())+'-'+str(tmp['pos'].max())+'\n')
    
    d[0].set_xlim(tmp['pos'].min()-0.01*lenR,tmp['pos'].max()+0.01*lenR)

In [ ]:
sns.histplot(data=[1], color=typec['Alpha'])

In [ ]:
sns.histplot(data=[1], color=typec['Beta'])

In [ ]:
sns.histplot(data=[1], color=typec['Delta'])

In [ ]:
sns.histplot(data=[1], color=typec['Duct'])

In [ ]:
sd_normalpancreas = []

In [ ]:
tmp = dmrs.loc[(dmrs['sig.comparison'].apply(lambda x:x[7*2:10*2])=='1|3|3|')&(dmrs['#Hypo']==1)&(dmrs['#Int']==0)].sort_values('meandiffabs', ascending=False).iloc[0]
ab = [tmp['chr'],tmp['start'],tmp['stop'],]
tmp = met.loc[(met['chr']==ab[0])&(met['pos']>=(int(ab[1])-1e3))&(met['pos']<=(int(ab[-1])+1e3))]
sd_normalpancreas.append(tmp)
plotIGV(tmp,0.7)
plt.savefig('./figures/ED5a.pdf', bbox_inches='tight')

In [ ]:
tmp = dmrs.loc[(dmrs['sig.comparison'].apply(lambda x:x[7*2:10*2])=='3|1|3|')&(dmrs['#Hypo']==1)&(dmrs['#Int']==0)].sort_values('meandiffabs', ascending=False).iloc[0]
ab = [tmp['chr'],tmp['start'],tmp['stop'],]
tmp = met.loc[(met['chr']==ab[0])&(met['pos']>=(int(ab[1])-1e3))&(met['pos']<=(int(ab[-1])+1e3))]
sd_normalpancreas.append(tmp)
plotIGV(tmp,0.7)
plt.savefig('./figures/ED5a-m.pdf', bbox_inches='tight')

In [ ]:
tmp = dmrs.loc[(dmrs['sig.comparison'].apply(lambda x:x[7*2:10*2])=='3|3|1|')&(dmrs['#Hypo']==1)&(dmrs['#Int']>=0)].sort_values('meandiffabs', ascending=False).iloc[0]
ab = [tmp['chr'],tmp['start'],tmp['stop'],]
tmp = met.loc[(met['chr']==ab[0])&(met['pos']>=(int(ab[1])-1e3))&(met['pos']<=(int(ab[-1])+1e3))]
sd_normalpancreas.append(tmp)
plotIGV(tmp,0.7)
plt.savefig('./figures/ED5a-r.pdf', bbox_inches='tight')

In [ ]:
sd_normalpancreas = pd.concat(sd_normalpancreas)
sd_normalpancreas_mean = sd_normalpancreas[['chr','pos']]
for k in ['Alpha','Beta','Delta']:
    sd_normalpancreas_mean[k] = sd_normalpancreas[sd_normalpancreas.columns[sd_normalpancreas.columns.str.contains(k)]].T.mean()
sd_normalpancreas_mean.to_csv('../SourceData/Fig.ED5a.txt', sep='\t', index=False)
sd_normalpancreas_mean